# DNAffinity: train and test models

Minimal example of the DNAffinity pipeline: load a preprocessed affinity table, build the physics-based features, train a model, and look at the predictions.

It uses the bundled **Gata4 uPBM** test case. Change `REGRESSOR` below to try any model from `dnaffinity/models/`: `random_forest`, `linear`, `ridge`, `svr`, `DFN`, `CNN1D`, `CNNAttention`, `Transformer`.

In [ ]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/PieterZanders/DNAffinity.git
    %cd DNAffinity
else:
    pass

In [ ]:
# Setup: make the dnaffinity package importable from the repo root
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from dnaffinity import Dataset, Model
from dnaffinity.common import default_params
from dnaffinity.plotting import (plot_feature_importances, plot_predictions,
                                 plot_training_curve)

np.random.seed(20)

print(f'Repo root: {REPO_ROOT}')

## 1. Load the data

Each row is a DNA probe (`ID_REF`), its measured binding affinity (`VALUE`), and a sample weight (`WEIGHT`).

In [2]:
PROTEIN = 'Gata4'

df_train = pd.read_csv(REPO_ROOT / 'data' / 'upbm' / PROTEIN / f'{PROTEIN}_training.txt', sep='\t')
print(f'{len(df_train)} probes')
df_train.head()

5511 probes


,ID_REF,VALUE,WEIGHT
0,ACTCAAAGAC,0.094644,1
1,ACCTATAAAG,0.070668,1
2,CCCGATAACA,0.342472,1
3,ATATCTAAGG,0.217709,1
4,TCTGAGACGG,0.061319,1


## 2. Build features and the model

`Dataset` turns the probe sequences into physics-based features (tetramer presence, average structural parameters, force-constant diagonals). `Model` picks the right input (flat features or per-window tensors) and builds the regressor, but does **not** train it yet — `print(model)` shows the split and the architecture.

For the PyTorch models, hyperparameters come from `default_params(REGRESSOR)` — the same defaults as `configs/{model}_config.json`. Override any of them in `params` below (e.g. fewer epochs for a quick notebook run).

In [ ]:
REGRESSOR = 'random_forest'   # try: 'DFN', 'CNN1D', 'CNNAttention', 'Transformer', 'ridge', ...

len_aln = len(df_train['ID_REF'].iloc[0])

dataset = Dataset(
    PROTEIN,
    df_train,
    model_target='octamers',
    chosen_features=['presence_tetramer', 'avg', 'diagonal_fce'],
    selected_tetramers=list(range(len_aln - 3)),
    weighted=True,
)

params = default_params(REGRESSOR)
params.update({'num_epochs': 20})   # keep PyTorch runs short in the notebook

model = Model(dataset, len_aln, regressor=REGRESSOR, training_set_size=0.9,
              weighted=True, pytorch_model_params=params)
print(model)

## 3. Train

Splits train/test and fits the regressor. PyTorch models show a progress bar with the training loss and the loss on the held-out split (monitoring only, it is not used for fitting).

In [ ]:
model.train()

### Training curve

Per-epoch loss for the PyTorch models; `model.history` is `None` for the scikit-learn models.

In [ ]:
if model.history is not None:
    plot_training_curve(model.history)
else:
    print(f'{REGRESSOR} is not trained by epochs, no training curve')

## 4. Test

`predict()` evaluates on the held-out test split (and on the training split for comparison). The scatter shows predicted vs measured affinity.

In [ ]:
model.predict()
print(f'test r2  = {model.r2}')
print(f'train r2 = {model.r2_train}')

plot_predictions(model.y_test, model.y_pred, model.r2,
                 y_train=model.y_train, y_train_pred=model.y_train_pred,
                 r2_train=model.r2_train)

## 5. Feature importances

Only available for models that expose them (random forest). For other regressors this cell prints a note and moves on.

In [ ]:
importances = plot_feature_importances(model.features, model.l)
if importances is not None:
    display(importances.head(10))
else:
    print(f'{REGRESSOR} does not provide feature importances')

## 6. Compare several models

Quick comparison on the same train/test split. PyTorch models run with few epochs here, so their scores are indicative only — use the scripts in `scripts/` with the full configs for real runs.

In [ ]:
results = {}
for name in ['random_forest', 'ridge', 'DFN']:
    params = default_params(name)
    params.update({'num_epochs': 20})
    m = Model(dataset, len_aln, regressor=name, training_set_size=0.9,
              weighted=True, pytorch_model_params=params)
    m.train()
    m.predict()
    results[name] = {'test r2': m.r2, 'train r2': m.r2_train}

pd.DataFrame(results).T